# Đánh giá Phase 2 trên ST-VQA — Kaggle T4 x2

Notebook này **không huấn luyện lại**. Nó dùng checkpoint `best_adapter.pt` đã có để đánh giá trên ST-VQA với bốn nhánh:

1. ảnh sạch + Qwen base;
2. ảnh sạch + adapter;
3. ảnh suy thoái + Qwen base;
4. ảnh suy thoái + adapter.

Metric chính là **ANLS**. Metric phụ gồm **Official VQA Accuracy, Exact Match và BLEU-1/2/3/4**.

## Trước khi chạy

- Settings → Accelerator → **GPU T4 x2**.
- Bật **Internet**.
- Add Input: `anhthu128/st-vqa-suy-thoi`.
- Add Input: dataset của bạn chứa đúng một file `best_adapter.pt` (thường là `embechuangoan/textvqa-feature-restore-v1`).


In [ ]:
import os, subprocess, sys
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

packages = [
    "transformers==4.57.6", "qwen-vl-utils==0.0.14", "accelerate==1.10.1",
    "datasets==4.1.1", "rapidfuzz==3.14.1", "nltk==3.9.1", "safetensors==0.5.3"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages], check=True)

repo = "/kaggle/working/textvqa-feature-restore-phase2"
if os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run(["git", "-C", repo, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ducdanh2304/textvqa-feature-restore-phase2.git", repo], check=True)
print("Chuẩn bị mã nguồn và thư viện: OK")


In [ ]:
from pathlib import Path
import pandas as pd
import torch

print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Hãy chọn GPU T4 x2 trong Settings"

input_root = Path("/kaggle/input")
checkpoints = sorted(input_root.rglob("best_adapter.pt"))
manifests = []
for path in input_root.rglob("final_main_manifest.csv"):
    try:
        columns = set(pd.read_csv(path, nrows=1).columns)
    except Exception:
        continue
    if {"image_id", "question_id", "answers_json", "source_row_index"}.issubset(columns):
        if (path.parent / "images/realistic_mix/L2").is_dir():
            manifests.append(path)

print("Checkpoint tìm thấy:")
for path in checkpoints: print(" -", path)
print("ST-VQA manifest tìm thấy:")
for path in manifests: print(" -", path)
assert len(checkpoints) == 1, "Cần attach đúng một dataset chứa best_adapter.pt"
assert len(manifests) == 1, "Cần attach đúng dataset anhthu128/st-vqa-suy-thoi"


## Chạy đánh giá

Mặc định lấy **500 câu cố định với seed 42**. Hai GPU chia đôi số câu. Cell có khả năng tiếp tục: nếu phiên bị ngắt, chạy lại sẽ giữ các dự đoán đã hoàn thành trong `/kaggle/working`.


In [ ]:
command = [
    sys.executable,
    "/kaggle/working/textvqa-feature-restore-phase2/run_stvqa_kaggle.py",
    "--sample-size", "500",
    "--seed", "42",
    "--include-clean",
]
subprocess.run(command, check=True)


In [ ]:
import json
from IPython.display import Image as DisplayImage, display

out = Path("/kaggle/working/stvqa_phase3_evaluation")
summary = json.loads((out / "summary.json").read_text(encoding="utf-8"))
display(pd.read_csv(out / "metrics_by_arm.csv").style.format({
    "anls": "{:.4f}", "vqa_accuracy": "{:.4f}", "exact_match": "{:.4f}",
    "bleu1": "{:.4f}", "bleu2": "{:.4f}", "bleu3": "{:.4f}", "bleu4": "{:.4f}"
}))
print("So sánh paired trên ảnh suy thoái:")
print(json.dumps(summary["paired_bootstrap"]["degraded"], ensure_ascii=False, indent=2))
display(DisplayImage(filename=str(out / "visualizations/all_metrics_by_arm.png")))
display(DisplayImage(filename=str(out / "visualizations/win_tie_loss_anls.png")))
display(DisplayImage(filename=str(out / "visualizations/qualitative_examples.png")))


## Tệp cần tải hoặc lưu thành Dataset

Sau khi hoàn tất, mở tab **Output** và lấy:

```text
/kaggle/working/stvqa_phase3_evaluation.zip
```

Bên trong có dự đoán từng câu, bảng metric, bootstrap CI, win/tie/loss và hình minh họa. Nhấn **Save Version** để Kaggle giữ kết quả.
